# Async Image Generation + Upscale Pipeline with Bedrock
This notebook provisions complete infrastructure for an async pipeline:
- API Gateway for job submission and status checking
- Step Functions for orchestration
- Lambda functions for prompt improvement, image generation, and image upscaling
- S3 for artifact storage

**Models** (all ACTIVE): Amazon Nova Lite (prompt), Stability Stable Image Core (image),
Stability Creative Upscale (upscale). The Stability image models are invoked via cross-region
inference profiles (the `us.` model ID prefix) and are synchronous `InvokeModel` calls — the
pipeline's async behavior comes from Step Functions + API Gateway job submission and polling.

> Note: This replaces the earlier Nova Canvas (image) + Nova Reel (video) design, since both of
> those models are retired. No ACTIVE Bedrock model performs image-to-video, so the pipeline now
> generates an image and then upscales it to ~4K.

In [21]:
import boto3
import json
import time

REGION = 'us-east-1'
PROJECT_NAME = 'bedrock-async-pipeline'
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

s3 = boto3.client('s3', region_name=REGION)
iam = boto3.client('iam', region_name=REGION)
lambda_client = boto3.client('lambda', region_name=REGION)
sfn = boto3.client('stepfunctions', region_name=REGION)
apigateway = boto3.client('apigatewayv2', region_name=REGION)

print(f"Account: {ACCOUNT_ID}, Region: {REGION}")

Account: 058264544288, Region: us-east-1


## 1. Create S3 Bucket for Artifacts

In [22]:
BUCKET_NAME = f"{PROJECT_NAME}-artifacts-{ACCOUNT_ID}"

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"✓ Created S3 bucket: {BUCKET_NAME}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"✓ S3 bucket already exists: {BUCKET_NAME}")

✓ Created S3 bucket: bedrock-async-pipeline-artifacts-058264544288


## 2. Create IAM Roles

In [23]:
# Lambda Role
lambda_role_name = f"{PROJECT_NAME}-lambda-role"
lambda_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

try:
    response = iam.create_role(RoleName=lambda_role_name, AssumeRolePolicyDocument=json.dumps(lambda_trust_policy))
    LAMBDA_ROLE_ARN = response['Role']['Arn']
    iam.attach_role_policy(RoleName=lambda_role_name, PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
    iam.attach_role_policy(RoleName=lambda_role_name, PolicyArn='arn:aws:iam::aws:policy/AmazonBedrockFullAccess')
    iam.attach_role_policy(RoleName=lambda_role_name, PolicyArn='arn:aws:iam::aws:policy/AmazonS3FullAccess')
    time.sleep(10)
    print("✓ Created Lambda role")
except iam.exceptions.EntityAlreadyExistsException:
    LAMBDA_ROLE_ARN = iam.get_role(RoleName=lambda_role_name)['Role']['Arn']
    print("✓ Lambda role exists")

# Step Functions Role
sfn_role_name = f"{PROJECT_NAME}-sfn-role"
sfn_trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "states.amazonaws.com"}, "Action": "sts:AssumeRole"}]
}

try:
    response = iam.create_role(RoleName=sfn_role_name, AssumeRolePolicyDocument=json.dumps(sfn_trust_policy))
    SFN_ROLE_ARN = response['Role']['Arn']
    iam.attach_role_policy(RoleName=sfn_role_name, PolicyArn='arn:aws:iam::aws:policy/AWSLambda_FullAccess')
    time.sleep(10)
    print("✓ Created Step Functions role")
except iam.exceptions.EntityAlreadyExistsException:
    SFN_ROLE_ARN = iam.get_role(RoleName=sfn_role_name)['Role']['Arn']
    print("✓ Step Functions role exists")

✓ Lambda role exists
✓ Step Functions role exists


## 3. Create Lambda Functions

In [24]:
import zipfile
from io import BytesIO

def create_lambda_function(name, code, handler, env_vars=None):
    """Create (or update) a Lambda from inline source, packaged as a ZIP."""
    function_name = f"{PROJECT_NAME}-{name}"

    zip_buffer = BytesIO()
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zip_file:
        zip_file.writestr('index.py', code.decode('utf-8'))
    zip_buffer.seek(0)
    zip_content = zip_buffer.read()

    try:
        response = lambda_client.create_function(
            FunctionName=function_name,
            Runtime='python3.12',
            Role=LAMBDA_ROLE_ARN,
            Handler=handler,
            Code={'ZipFile': zip_content},
            Timeout=300,
            MemorySize=512,
            Environment={'Variables': env_vars or {}}
        )
        print(f"✓ Created Lambda: {function_name}")
        return response['FunctionArn']
    except lambda_client.exceptions.ResourceConflictException:
        lambda_client.update_function_code(FunctionName=function_name, ZipFile=zip_content)
        response = lambda_client.get_function(FunctionName=function_name)
        print(f"✓ Updated Lambda: {function_name}")
        return response['Configuration']['FunctionArn']

In [25]:
# Lambda 1: Improve Prompt (Amazon Nova Lite)
improve_prompt_code = b'''
import json
import boto3

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

def lambda_handler(event, context):
    prompt = event['prompt']
    body = {
        "messages": [{"role": "user", "content": [{"text": f"Improve this image generation prompt: {prompt}"}]}],
        "inferenceConfig": {"maxTokens": 200, "temperature": 0.7}
    }
    response = bedrock.converse(modelId='us.amazon.nova-lite-v1:0', **body)
    improved = response['output']['message']['content'][0]['text']
    return {'prompt': prompt, 'improved_prompt': improved, 'job_id': event['job_id']}
'''

IMPROVE_PROMPT_ARN = create_lambda_function('improve-prompt', improve_prompt_code, 'index.lambda_handler')

✓ Updated Lambda: bedrock-async-pipeline-improve-prompt


In [26]:
# Lambda 2: Generate Image (Stability Stable Image Core)
generate_image_code = b'''
import json, boto3, base64, os

# Stable Image Core is a text-to-image generator available as an on-demand BASE model
# in us-west-2 (there is no 'us.' inference profile for it), so this client targets us-west-2.
bedrock = boto3.client('bedrock-runtime', region_name='us-west-2')
s3 = boto3.client('s3')

def lambda_handler(event, context):
    prompt = event['improved_prompt']
    job_id = event['job_id']
    bucket = os.environ['BUCKET_NAME']

    # Stability uses a flat body (no taskType wrapper)
    body = {
        "prompt": prompt,
        "aspect_ratio": "1:1",
        "output_format": "png",
        "seed": 42
    }

    response = bedrock.invoke_model(modelId='stability.stable-image-core-v1:1', body=json.dumps(body))
    result = json.loads(response['body'].read())

    if result.get('finish_reasons', [None])[0]:
        raise Exception(f"Image generation filtered: {result['finish_reasons'][0]}")

    image_b64 = result['images'][0]
    s3.put_object(Bucket=bucket, Key=f"{job_id}/image.png", Body=base64.b64decode(image_b64))

    # NOTE: do NOT return image_b64 - Step Functions caps state payloads at 256KB and a
    # base64 PNG exceeds that (States.DataLimitExceeded). The next stage reads it from S3.
    return {'job_id': job_id, 'improved_prompt': prompt, 'image_s3_key': f"{job_id}/image.png"}
'''

GENERATE_IMAGE_ARN = create_lambda_function('generate-image', generate_image_code, 'index.lambda_handler', {'BUCKET_NAME': BUCKET_NAME})

✓ Updated Lambda: bedrock-async-pipeline-generate-image


In [27]:
# Lambda 3: Upscale Image (Stability Creative Upscale)
upscale_image_code = b'''
import json, boto3, base64, os

# Creative Upscale runs via the us.stability inference profile (available in us-west-2).
bedrock = boto3.client('bedrock-runtime', region_name='us-west-2')
s3 = boto3.client('s3')

def lambda_handler(event, context):
    job_id = event['job_id']
    bucket = os.environ['BUCKET_NAME']

    # The image is read from S3 (the generate stage does not pass bytes through Step
    # Functions, which caps state payloads at 256KB).
    obj = s3.get_object(Bucket=bucket, Key=event['image_s3_key'])
    image_b64 = base64.b64encode(obj['Body'].read()).decode('utf-8')

    body = {
        "image": image_b64,
        "prompt": event.get('improved_prompt', 'high quality, sharp detail, professional photography'),
        "creativity": 0.3,
        # Creative Upscale produces ~4K output; a PNG response can exceed Bedrock's 16MB
        # InvokeModel response limit, so request JPEG (much smaller for photographic images).
        "output_format": "jpeg"
    }

    response = bedrock.invoke_model(modelId='us.stability.stable-creative-upscale-v1:0', body=json.dumps(body))
    result = json.loads(response['body'].read())

    if result.get('finish_reasons', [None])[0]:
        raise Exception(f"Upscale filtered: {result['finish_reasons'][0]}")

    upscaled_b64 = result['images'][0]
    s3.put_object(Bucket=bucket, Key=f"{job_id}/upscaled.jpg", Body=base64.b64decode(upscaled_b64))

    return {'job_id': job_id, 'image_s3_key': event.get('image_s3_key'),
            'upscaled_s3_key': f"{job_id}/upscaled.jpg", 'status': 'COMPLETED'}
'''

UPSCALE_IMAGE_ARN = create_lambda_function('upscale-image', upscale_image_code, 'index.lambda_handler', {'BUCKET_NAME': BUCKET_NAME})

✓ Updated Lambda: bedrock-async-pipeline-upscale-image


## 4. Create Step Functions State Machine

In [28]:
state_machine_name = f"{PROJECT_NAME}-workflow"

definition = {
    "Comment": "Async image generation + upscale pipeline",
    "StartAt": "ImprovePrompt",
    "States": {
        "ImprovePrompt": {"Type": "Task", "Resource": IMPROVE_PROMPT_ARN, "Next": "GenerateImage"},
        "GenerateImage": {"Type": "Task", "Resource": GENERATE_IMAGE_ARN, "Next": "UpscaleImage"},
        "UpscaleImage": {"Type": "Task", "Resource": UPSCALE_IMAGE_ARN, "End": True}
    }
}

try:
    response = sfn.create_state_machine(
        name=state_machine_name, definition=json.dumps(definition),
        roleArn=SFN_ROLE_ARN, type='STANDARD'
    )
    STATE_MACHINE_ARN = response['stateMachineArn']
    print("✓ Created State Machine")
except sfn.exceptions.StateMachineAlreadyExists:
    machines = sfn.list_state_machines()['stateMachines']
    STATE_MACHINE_ARN = next(m['stateMachineArn'] for m in machines if m['name'] == state_machine_name)
    sfn.update_state_machine(stateMachineArn=STATE_MACHINE_ARN, definition=json.dumps(definition))
    print("✓ Updated State Machine")

print(f"State Machine ARN: {STATE_MACHINE_ARN}")

✓ Created State Machine
State Machine ARN: arn:aws:states:us-east-1:058264544288:stateMachine:bedrock-async-pipeline-workflow


## 5. Create API Lambda Functions

In [29]:
# Start Execution Lambda
start_execution_code = f'''
import json, boto3, uuid
sfn = boto3.client('stepfunctions')

def lambda_handler(event, context):
    body = json.loads(event.get('body', '{{}}'))
    prompt = body.get('prompt', '')
    if not prompt:
        return {{'statusCode': 400, 'body': json.dumps({{'error': 'prompt required'}})}}

    job_id = str(uuid.uuid4())
    response = sfn.start_execution(
        stateMachineArn='{STATE_MACHINE_ARN}',
        input=json.dumps({{'prompt': prompt, 'job_id': job_id}})
    )
    return {{'statusCode': 200, 'body': json.dumps({{'job_id': job_id, 'execution_arn': response['executionArn']}})}}
'''.encode()

START_EXECUTION_ARN = create_lambda_function('start-execution', start_execution_code, 'index.lambda_handler')

✓ Updated Lambda: bedrock-async-pipeline-start-execution


In [30]:
# Check Status Lambda
check_status_code = b'''
import json, boto3
sfn = boto3.client('stepfunctions')

def lambda_handler(event, context):
    execution_arn = event['queryStringParameters'].get('execution_arn')
    if not execution_arn:
        return {'statusCode': 400, 'body': json.dumps({'error': 'execution_arn required'})}

    response = sfn.describe_execution(executionArn=execution_arn)
    result = {'status': response['status'], 'start_date': response['startDate'].isoformat()}

    if response['status'] == 'SUCCEEDED':
        output = json.loads(response.get('output', '{}'))
        result['artifacts'] = {'image': output.get('image_s3_key'), 'upscaled': output.get('upscaled_s3_key')}

    return {'statusCode': 200, 'body': json.dumps(result)}
'''

CHECK_STATUS_ARN = create_lambda_function('check-status', check_status_code, 'index.lambda_handler')

✓ Updated Lambda: bedrock-async-pipeline-check-status


## 6. Create API Gateway

In [31]:
api_name = f"{PROJECT_NAME}-api"

# Idempotent: reuse an existing API of this name instead of creating duplicates on re-run.
existing = [a for a in apigateway.get_apis()['Items'] if a['Name'] == api_name]
if existing:
    api_id = existing[0]['ApiId']
    print(f"✓ Reusing existing API: {api_id}")
else:
    api = apigateway.create_api(Name=api_name, ProtocolType='HTTP', Target=START_EXECUTION_ARN)
    api_id = api['ApiId']
    print(f"✓ Created API: {api_id}")

# Grant API Gateway permission to invoke each Lambda.
# Use a StatementId that is UNIQUE PER API ID so re-runs (which may create a new API)
# always add a matching permission. A fixed StatementId would silently fail on re-run
# and cause HTTP 500 (AccessDenied) when the new API tries to invoke the Lambda.
def grant_invoke(function_arn, suffix):
    fn = function_arn.split(':')[-1]
    try:
        lambda_client.add_permission(
            FunctionName=fn,
            StatementId=f"apigw-{api_id}-{suffix}",
            Action='lambda:InvokeFunction',
            Principal='apigateway.amazonaws.com',
            SourceArn=f"arn:aws:execute-api:{REGION}:{ACCOUNT_ID}:{api_id}/*/*"
        )
        print(f"✓ Granted invoke on {fn} for API {api_id}")
    except lambda_client.exceptions.ResourceConflictException:
        print(f"✓ Invoke permission already present on {fn} for API {api_id}")

grant_invoke(START_EXECUTION_ARN, 'start')
grant_invoke(CHECK_STATUS_ARN, 'status')

# Ensure integrations exist (create only if missing).
existing_integrations = {i['IntegrationUri']: i['IntegrationId']
                         for i in apigateway.get_integrations(ApiId=api_id)['Items']
                         if i.get('IntegrationUri')}

def ensure_integration(uri):
    if uri in existing_integrations:
        return existing_integrations[uri]
    integ = apigateway.create_integration(
        ApiId=api_id, IntegrationType='AWS_PROXY', IntegrationUri=uri, PayloadFormatVersion='2.0'
    )
    return integ['IntegrationId']

start_integration_id = ensure_integration(START_EXECUTION_ARN)
status_integration_id = ensure_integration(CHECK_STATUS_ARN)

# Ensure routes exist (create only if missing).
existing_routes = {r['RouteKey'] for r in apigateway.get_routes(ApiId=api_id)['Items']}
if 'POST /generate' not in existing_routes:
    apigateway.create_route(ApiId=api_id, RouteKey='POST /generate', Target=f"integrations/{start_integration_id}")
if 'GET /status' not in existing_routes:
    apigateway.create_route(ApiId=api_id, RouteKey='GET /status', Target=f"integrations/{status_integration_id}")

# Ensure the 'prod' stage exists.
existing_stages = {s['StageName'] for s in apigateway.get_stages(ApiId=api_id)['Items']}
if 'prod' not in existing_stages:
    apigateway.create_stage(ApiId=api_id, StageName='prod', AutoDeploy=True)

API_ENDPOINT = f"https://{api_id}.execute-api.{REGION}.amazonaws.com/prod"
print(f"✓ API ready: {API_ENDPOINT}")

✓ Reusing existing API: 61q9v00d3k
✓ Invoke permission already present on bedrock-async-pipeline-start-execution for API 61q9v00d3k
✓ Invoke permission already present on bedrock-async-pipeline-check-status for API 61q9v00d3k
✓ API ready: https://61q9v00d3k.execute-api.us-east-1.amazonaws.com/prod


## 7. Test the Pipeline

In [32]:
import requests

# Start a job
response = requests.post(f"{API_ENDPOINT}/generate", json={"prompt": "A serene mountain landscape at sunset"})
job_data = response.json()
print(f"Job started: {job_data}")

execution_arn = job_data['execution_arn']
job_id = job_data['job_id']

Job started: {'job_id': '2be47eeb-22f3-44a1-a6ee-3d1740d58a7a', 'execution_arn': 'arn:aws:states:us-east-1:058264544288:execution:bedrock-async-pipeline-workflow:91c59568-85ff-4083-94c8-9cfc689b5a41'}


In [41]:
# Check status (poll until SUCCEEDED)
response = requests.get(f"{API_ENDPOINT}/status", params={"execution_arn": execution_arn})
status = response.json()
print(json.dumps(status, indent=2))

{
  "status": "SUCCEEDED",
  "start_date": "2026-10-01T04:33:32.812000+00:00",
  "artifacts": {
    "image": "2be47eeb-22f3-44a1-a6ee-3d1740d58a7a/image.png",
    "upscaled": "2be47eeb-22f3-44a1-a6ee-3d1740d58a7a/upscaled.jpg"
  }
}


## Summary

Infrastructure created:
- S3 bucket for artifacts (`<job_id>/image.png`, `<job_id>/upscaled.jpg`)
- 5 Lambda functions (improve prompt, generate image, upscale image, start execution, check status)
- Step Functions state machine: `ImprovePrompt → GenerateImage → UpscaleImage`
- API Gateway with 2 endpoints

Models (all ACTIVE):
- Prompt: `us.amazon.nova-lite-v1:0`
- Image: `us.stability.stable-image-core-v1:1`
- Upscale: `us.stability.stable-creative-upscale-v1:0`

Usage:
```bash
# Start job
curl -X POST {API_ENDPOINT}/generate -d '{"prompt":"your text"}'

# Check status
curl {API_ENDPOINT}/status?execution_arn=<arn>
```